# The Distance Formula — Sparse Symbolic Regression via FISTA

This notebook recovers the distance formula between two points, $(x_1,y_1)$ and $(x_2,y_2)$, from
data alone, using FISTA for optimization.

In [17]:
import numpy as np

In [18]:
# FISTA: soft-thresholding + Nesterov momentum, with a backtracking step size since the
# hypothesis below is nonlinear.
# 'penalize_mask' skips the L1 penalty on xi0 and xi5 (the sqrt's own scale) -- shrinking those
# would just shrink the whole formula, not pick which terms to keep.

def soft_threshold(z, thresh):
    return np.sign(z) * np.maximum(np.abs(z) - thresh, 0.0)

def fista(f, grad_fn, x0, lam, penalize_mask=None, max_iter=8000, L0=10.0, eta=2.0, tol=1e-14):
    n = len(x0)
    if penalize_mask is None:
        penalize_mask = np.ones(n)
    x = np.array(x0, dtype=float)
    y = x.copy()
    t = 1.0
    L = L0
    for _ in range(max_iter):
        fy, gy = f(y), grad_fn(y)
        L = max(L / eta, L0 * 1e-6)
        while True:
            step = 1.0 / L
            raw = y - step * gy
            x_new = raw.copy()
            idx = penalize_mask > 0
            x_new[idx] = soft_threshold(raw[idx], lam * step * penalize_mask[idx])
            diff = x_new - y
            Q = fy + np.dot(gy, diff) + (L / 2.0) * np.dot(diff, diff)
            if f(x_new) <= Q + 1e-12 or L > 1e18:
                break
            L *= eta
        t_new = (1 + np.sqrt(1 + 4 * t * t)) / 2
        y = x_new + ((t - 1) / t_new) * (x_new - x)
        if np.linalg.norm(x_new - x) < tol:
            x = x_new
            break
        x, t = x_new, t_new
    return x

In [19]:
# Data generation. The labels are computed straight from the true formula; 
# the "discovery" here is about recovering the right
# sparse combination of candidate terms from those examples, not about avoiding circular labels.
# We skip the single point where the two points coincide (x1,y1)=(x2,y2), since the square root's
# slope is infinite there.

def make_dataset(n, seed):
    rng = np.random.default_rng(seed)
    pts = []
    while len(pts) < n:
        x1, y1, x2, y2 = rng.uniform(-5, 5, 4)
        if (x1 - x2) ** 2 + (y1 - y2) ** 2 > 1e-3:
            pts.append((x1, y1, x2, y2))
    arr = np.array(pts)
    x1, y1, x2, y2 = arr[:, 0], arr[:, 1], arr[:, 2], arr[:, 3]
    d = np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)
    return x1, y1, x2, y2, d

x1_tr, y1_tr, x2_tr, y2_tr, d_tr = make_dataset(2000, seed=2)
x1_te, y1_te, x2_te, y2_te, d_te = make_dataset(500, seed=3)
print(f"train: {len(d_tr)}  test: {len(d_te)}")

train: 2000  test: 500


In [20]:
# Hypothesis: an outer linear part in (x1,y1,x2,y2) plus the square root of a full
# degree-two monomial basis in the same four variables (21 candidate terms total; the true
# formula only needs 7 of them).
# xi layout: 0 const, 1 x1, 2 y1, 3 x2, 4 y2, 5 sqrt-coeff (all outside the root), then inside S:
# 6 const, 7 x1, 8 y1, 9 x2, 10 y2, 11 x1^2, 12 y1^2, 13 x2^2, 14 y2^2,
# 15 x1*y1, 16 x1*x2, 17 x1*y2, 18 y1*x2, 19 y1*y2, 20 x2*y2

N = 21

def S_terms(x1, y1, x2, y2):
    return {
        6: np.ones_like(x1), 7: x1, 8: y1, 9: x2, 10: y2,
        11: x1**2, 12: y1**2, 13: x2**2, 14: y2**2,
        15: x1*y1, 16: x1*x2, 17: x1*y2, 18: y1*x2, 19: y1*y2, 20: x2*y2,
    }

def compute_S(xi, x1, y1, x2, y2):
    S = np.zeros_like(x1)
    for k, v in S_terms(x1, y1, x2, y2).items():
        S = S + xi[k] * v
    return S

def hypothesis(xi, x1, y1, x2, y2):
    S = compute_S(xi, x1, y1, x2, y2)
    sqrtS = np.sqrt(np.maximum(S, 1e-10))
    return xi[0] + xi[1]*x1 + xi[2]*y1 + xi[3]*x2 + xi[4]*y2 + xi[5]*sqrtS

def objective(xi, x1, y1, x2, y2, t):
    return np.mean(0.5 * (hypothesis(xi, x1, y1, x2, y2) - t) ** 2)

In [21]:
# Gradient, derived by hand via the chain rule through sqrt(S) (replaces autograd).
# xhat = xi0 + xi1*x1 + ... + xi4*y2 + xi5*sqrt(S)
#   xi0..xi4 sit outside the root -> their derivative is just the matching input (or 1 for xi0)
#   xi5 multiplies sqrt(S) directly -> its derivative is sqrt(S) itself
#   xi6..xi20 sit inside S -> derivative is xi5 * (dS/dxi_k) / (2*sqrt(S))
# We floor S at a small positive number so we never divide by (or take the root of) zero.

def grad(xi, x1, y1, x2, y2, t):
    S = compute_S(xi, x1, y1, x2, y2)
    mask = (S > 1e-10).astype(float)
    sqrtS = np.sqrt(np.maximum(S, 1e-10))
    resid = xi[0] + xi[1]*x1 + xi[2]*y1 + xi[3]*x2 + xi[4]*y2 + xi[5]*sqrtS - t

    g = np.zeros(N)
    g[0] = np.mean(resid)
    g[1] = np.mean(resid * x1)
    g[2] = np.mean(resid * y1)
    g[3] = np.mean(resid * x2)
    g[4] = np.mean(resid * y2)
    g[5] = np.mean(resid * sqrtS)
    for k, v in S_terms(x1, y1, x2, y2).items():
        g[k] = np.mean(resid * xi[5] * mask * v / (2 * sqrtS))
    return g

In [22]:
# Fit with a few random restarts and keep the best.
# xi0 and xi5 are exempt from the L1 penalty, same reasoning as before: they set the overall
# scale, not which terms are active.

penalize_mask = np.ones(N)
penalize_mask[0] = 0.0
penalize_mask[5] = 0.0

def fit_distance(n_restarts=6, lam=1e-4, max_iter=8000):
    best_xi, best_err = None, np.inf
    for seed in range(n_restarts):
        rng = np.random.default_rng(seed)
        xi0 = rng.normal(0, 0.5, N)
        obj = lambda xi: objective(xi, x1_tr, y1_tr, x2_tr, y2_tr, d_tr)
        gfn = lambda xi: grad(xi, x1_tr, y1_tr, x2_tr, y2_tr, d_tr)
        xi_fit = fista(obj, gfn, xi0, lam=lam, penalize_mask=penalize_mask, max_iter=max_iter, L0=10.0)
        err = obj(xi_fit)
        if err < best_err:
            best_err, best_xi = err, xi_fit
    return best_xi, best_err

xi, train_err = fit_distance()
test_err = objective(xi, x1_te, y1_te, x2_te, y2_te, d_te)
print("train fit error:", train_err)
print("test  fit error:", test_err)

train fit error: 1.5160498485922586e-09
test  fit error: 1.8279324640202462e-09


In [25]:
# Scale normalization + hard thresholding. Many different sets of numbers here would give the
# same predictions - if you multiply everything in S by some number squared, and divide xi5
# by that same number, the answer doesn't change. To make our numbers comparable to the known
# correct answer, we fix this: we rescale everything so that the coefficient of x1^2 in S comes
# out to exactly 1, its true value.

def normalize(xi):
    xi = xi.copy()
    c = np.sqrt(xi[11])
    xi[6:21] /= c ** 2
    xi[5] *= c
    return xi

def threshold(xi, tol=1e-2):
    xi = xi.copy()
    xi[np.abs(xi) < tol] = 0.0
    return xi

xi_final = threshold(normalize(xi))

In [26]:
# Final result: the recovered formula should match d = sqrt((x2-x1)^2 + (y2-y1)^2).

names = {0: 'xi0 (const, outside)', 5: 'xi5 (sqrt coeff)', 11: 'xi11 (x1^2 in S)',
         12: 'xi12 (y1^2 in S)', 13: 'xi13 (x2^2 in S)', 14: 'xi14 (y2^2 in S)',
         16: 'xi16 (x1*x2 in S)', 19: 'xi19 (y1*y2 in S)'}
exact = {5: 1.0, 11: 1.0, 12: 1.0, 13: 1.0, 14: 1.0, 16: -2.0, 19: -2.0}

print(f"{'Coefficient':<24}{'Recovered':<14}{'Exact':<10}")
for k in sorted(names):
    print(f"{names[k]:<24}{xi_final[k]:<14.4f}{exact.get(k, 0.0):<10.2f}")

remaining = [i for i in range(N) if i not in names]
print()
print("All remaining coefficients:", xi_final[remaining])

Coefficient             Recovered     Exact     
xi0 (const, outside)    0.0000        0.00      
xi5 (sqrt coeff)        1.0000        1.00      
xi11 (x1^2 in S)        1.0000        1.00      
xi12 (y1^2 in S)        1.0000        1.00      
xi13 (x2^2 in S)        1.0000        1.00      
xi14 (y2^2 in S)        1.0000        1.00      
xi16 (x1*x2 in S)       -2.0000       -2.00     
xi19 (y1*y2 in S)       -2.0000       -2.00     

All remaining coefficients: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
